# SPEI Fraud Detection - Baseline Model

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from xgboost import XGBClassifier
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.metrics import(
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve
)
import shap
from src.features.load_data import load_fct_features, load_fraud_typology

In [ ]:
df = load_fct_features()
typology_df = load_fraud_typology()
df = pd.merge(df, typology_df, how='left', on='transaction_id')
df.shape

In [ ]:
df = df.sort_values(by='transaction_date').reset_index(drop=True)
y = df['is_fraud'].astype(int)

cols_to_drop = [
    'transaction_id',
    'sender_account_id',
    'receiver_account_id',
    'transaction_date',
    'is_fraud',
    'fraud_typology'
]

X = df.drop(columns=cols_to_drop)
print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Target Distribution: {y.value_counts(normalize=True).round(4)}')

In [ ]:
sklearn.set_config(transform_output='pandas')

preprocessor = ColumnTransformer(
    transformers=[
        (
            'ordinal_segment', OrdinalEncoder(categories=[['dormant', 'light', 'regular', 'heavy', 'power']]),
            ['sender_activity_segment']
        ),
        (
            'onehot_concept',
            OneHotEncoder(
                max_categories=16,
                handle_unknown='infrequent_if_exist',
                sparse_output=False
            ),
            ['concept_pago']
        ),
    ],
    remainder='passthrough',
    verbose_feature_names_out = False
)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(scale_pos_weight=67, random_state=42, n_jobs=-1, eval_metric='aucpr')),
])
pipeline

In [ ]:
#80/10/10 split
n = len(df)
train_end = int(n * .8)
val_end = int(n * .9)

X_train, X_val, X_test = X.iloc[:train_end], X.iloc[train_end:val_end], X.iloc[val_end:]
y_train, y_val, y_test = y.iloc[:train_end], y.iloc[train_end:val_end], y.iloc[val_end:]


In [ ]:
test_df = df.iloc[val_end:]
typology_counts = test_df['fraud_typology'].value_counts(dropna=True)
print('Per-typology counts in test:')
print(typology_counts)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_val_pred = pipeline.predict(X_val)
y_val_proba = pipeline.predict_proba(X_val)[:, 1]

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
print("Confusion matrix on val:")
print(pd.DataFrame(cm,
    index=['actual_legit', 'actual_fraud'],
    columns=['pred_legit', 'pred_fraud']))

In [ ]:
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
auc_roc = roc_auc_score(y_val, y_val_proba)
auc_pr = average_precision_score(y_val, y_val_proba)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")
print(f"AUC-ROC: {auc_roc:.3f}")
print(f"AUC-PR:{auc_pr:.3f}")

In [ ]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / 'artifacts' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODELS_DIR / 'baselinexgb.pkl'
joblib.dump(pipeline, model_path)

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)

f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]
best_precision = precisions[best_idx]
best_recall = recalls[best_idx]

print(f"Best F1: {best_f1:.3f}")
print(f"At threshold: {best_threshold:.4f}")
print(f"  Precision: {best_precision:.3f}")
print(f"  Recall:    {best_recall:.3f}")

In [ ]:
plt.figure(figsize= (7, 5))
plt.plot(recalls, precisions)
plt.scatter(best_recall, best_precision, color='red', label='Max F1')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.plot(thresholds, f1_scores, label='F1')
plt.axvline(best_threshold, color='gray', linestyle='--', label='Max F1 threshold')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Metrics vs Threshold')
plt.legend()
plt.show()

In [ ]:
# F2
beta = 2
f2_scores = (1 + beta**2) * (precisions[:-1] * recalls[:-1]) / (beta**2 * precisions[:-1] + recalls[:-1] + 1e-9)
best_f2_idx = np.argmax(f2_scores)

mask = recalls[:-1] >= 0.80
constrained_idx = np.where(mask)[0][np.argmax(precisions[:-1][mask])]

print(f"Max F1: threshold={best_threshold:.3f}, P={best_precision:.3f}, R={best_recall:.3f}")
print(f"Max F2: threshold={thresholds[best_f2_idx]:.3f}, P={precisions[best_f2_idx]:.3f}, R={recalls[best_f2_idx]:.3f}")
print(f"R>=0.80: threshold={thresholds[constrained_idx]:.3f}, P={precisions[constrained_idx]:.3f}, R={recalls[constrained_idx]:.3f}")

In [ ]:
operating_threshold = 0.808
y_val_pred_tuned = (y_val_proba >= operating_threshold).astype(int)

In [ ]:
print(classification_report(y_val, y_val_pred_tuned, target_names=['legit', 'fraud']))
print(f"AUC-ROC: {roc_auc_score(y_val, y_val_proba):.3f}")
print(f"AUC-PR:  {average_precision_score(y_val, y_val_proba):.3f}")

In [ ]:
val_typology = df.iloc[train_end:val_end]['fraud_typology'].values

for typology in ['app_fraud', 'ato', 'mule_to_mule', 'smurfing', 'business_impersonation']:
    mask = val_typology == typology
    n_total = mask.sum()
    n_caught = y_val_pred_tuned[mask].sum()
    print(f"{typology}: {n_caught}/{n_total} ({n_caught/n_total:.1%})")

### Test set:

In [ ]:
y_test_pred = pipeline.predict(X_test)
y_test_proba = pipeline.predict_proba(X_test)[:, 1]
y_test_pred_tuned = (y_test_proba >= operating_threshold).astype(int)

print(classification_report(y_test, y_test_pred_tuned, target_names=['legit', 'fraud']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_test_proba):.3f}")
print(f"AUC-PR:  {average_precision_score(y_test, y_test_proba):.3f}")

In [ ]:
test_typology = df.iloc[val_end:]['fraud_typology'].values

for typology in ['app_fraud', 'ato', 'mule_to_mule', 'smurfing', 'business_impersonation']:
    mask = test_typology == typology
    n_total = mask.sum()
    n_caught = y_test_pred_tuned[mask].sum()
    print(f"{typology}: {n_caught}/{n_total} ({n_caught/n_total:.1%})")

In [ ]:
shap.initjs()
X_val_sample = X_val.sample(5000, random_state=42)
X_val_preprocessed = pipeline.named_steps['preprocessor'].transform(X_val_sample)

explainer = shap.TreeExplainer(pipeline.named_steps['model'])
shap_values = explainer(X_val_preprocessed)

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
fraud_idx_in_sample = np.where(pipeline.predict(X_val_sample) == 1)[0][0]
shap.plots.waterfall(shap_values[fraud_idx_in_sample])

In [ ]:
MODELS_DIR = Path.cwd().parent / 'artifacts' / 'models'

joblib.dump(pipeline, MODELS_DIR / 'baseline_xgb_tuned.pkl')
joblib.dump(operating_threshold, MODELS_DIR / 'operating_threshold.pkl')